# 銀化分類器 評価レポート

修論・学会発表用のFigureを生成するノートブック。

**前提:** `python scripts/03_run_baseline.py` を実行して `models/evaluation_report.json` ができている状態。

**生成される図:**
1. 主要指標サマリー（棒グラフ）
2. Precision-Recall 曲線（不均衡データでの本命）
3. ROC 曲線（慣習）
4. コスト-閾値曲線（FN:FP=10:1）
5. 分布シフト評価（正例比率の変動に対する頑健性）

参照: ghmagazine/evaluation_book (Apache-2.0) 第3章

## セットアップ

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

# silvering-classifier ルートを path に追加
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.evaluate import FoldPredictions
from src.visualize import (
    plot_cost_threshold_curve,
    plot_distribution_shift,
    plot_metric_summary_bar,
    plot_pr_curve,
    plot_roc_curve,
)

## 評価レポートを読み込む

In [ ]:
report_path = ROOT / "models" / "evaluation_report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))

preds = FoldPredictions(
    y_true=np.array(report["predictions"]["y_true"]),
    y_pred=np.array(report["predictions"]["y_pred"]),
    y_proba=np.array(report["predictions"]["y_proba"]),
)

metrics = report["metrics_at_0.5"]
cost = report["cost_optimization"]
shift = report["distribution_shift"]

print(f"サンプル数: {len(preds.y_true)}")
print(f"正例(smolt)率: {preds.y_true.mean():.2%}")
print(f"主指標 MCC: {metrics['mcc']:.3f}, PR-AUC: {metrics['pr_auc']:.3f}")

## Figure 1: 主要指標サマリー

**修論の使い所:** Results 章の冒頭、結果の俯瞰として配置。

緑(MCC/PR-AUC)が主指標、青(ROC-AUC/G-Mean)が補助指標、灰(accuracy/precision/recall/f1)が慣習指標。

In [ ]:
fig = plot_metric_summary_bar(metrics)
fig.savefig("figures/fig1_metric_summary.png", dpi=200, bbox_inches="tight")
fig.savefig("figures/fig1_metric_summary.pdf", bbox_inches="tight")

## Figure 2: Precision-Recall 曲線

**修論の使い所:** 不均衡データであることを示した後、なぜPR-AUCを主指標にしたかのdefendに使う。

PR-AUCがbaseline(=prevalence)を大きく上回っていれば、モデルが「ベースよりも上手く正例を識別できている」証拠になる。

In [ ]:
fig = plot_pr_curve(preds)
fig.savefig("figures/fig2_pr_curve.png", dpi=200, bbox_inches="tight")
fig.savefig("figures/fig2_pr_curve.pdf", bbox_inches="tight")

## Figure 3: ROC 曲線

**修論の使い所:** PR曲線の補助として併記。比較対象の論文との対応関係を示しやすい。

In [ ]:
fig = plot_roc_curve(preds)
fig.savefig("figures/fig3_roc_curve.png", dpi=200, bbox_inches="tight")
fig.savefig("figures/fig3_roc_curve.pdf", bbox_inches="tight")

## Figure 4: コスト考慮型閾値最適化

**修論の使い所:** Discussion章で「養殖場でのデプロイ」を論じる箇所に置く。

左図: 総コストが閾値とともにどう変化するか。最適閾値を明示。  
右図: FN/FP の構成。閾値を下げると FN が減って FP が増えるトレードオフが見える。

**FN:FP=10:1 はあくまで仮定**。修論では sensitivity analysis を別表として添えると審査員突っ込みづらい。

In [ ]:
fig = plot_cost_threshold_curve(cost)
fig.savefig("figures/fig4_cost_threshold.png", dpi=200, bbox_inches="tight")
fig.savefig("figures/fig4_cost_threshold.pdf", bbox_inches="tight")

print(f"デフォルト閾値 0.5 → Recall: {cost['default_metrics']['recall']:.3f}, FN: {cost['default_metrics']['fn']}")
print(f"最適閾値 {cost['best_threshold']:.3f} → Recall: {cost['optimal_metrics']['recall']:.3f}, FN: {cost['optimal_metrics']['fn']}")

## Figure 5: 分布シフト評価

**修論の使い所:** Discussion章「養殖場ごとに parr/smolt 比率が違うが、本モデルは頑健か？」の defend に使う。

- ROC-AUCが横ばい → 順位付け能力は分布に依存しない
- Accuracyが大きく動く → 「accuracy だけ報告するのは不適切」の根拠

In [ ]:
fig = plot_distribution_shift(shift)
fig.savefig("figures/fig5_distribution_shift.png", dpi=200, bbox_inches="tight")
fig.savefig("figures/fig5_distribution_shift.pdf", bbox_inches="tight")

## 修論本文用：指標サマリー表（LaTeX）

そのままコピペできる形で表を出力。

In [ ]:
lines = [
    r"\begin{tabular}{lr}",
    r"\toprule",
    r"Metric & Score \\",
    r"\midrule",
    f"Accuracy & {metrics['accuracy']:.3f} \\\\",
    f"Precision & {metrics['precision']:.3f} \\\\",
    f"Recall (Sensitivity) & {metrics['recall']:.3f} \\\\",
    f"Specificity & {metrics['specificity']:.3f} \\\\",
    f"F1-score & {metrics['f1']:.3f} \\\\",
    f"G-Mean & {metrics['g_mean']:.3f} \\\\",
    f"\\textbf{{MCC}} & \\textbf{{{metrics['mcc']:.3f}}} \\\\",
    f"ROC-AUC & {metrics['roc_auc']:.3f} \\\\",
    f"\\textbf{{PR-AUC}} & \\textbf{{{metrics['pr_auc']:.3f}}} \\\\",
    r"\bottomrule",
    r"\end{tabular}",
]
print("\n".join(lines))

## まとめ

出力されたFigure (`figures/`):
- `fig1_metric_summary.{png,pdf}` — 主要指標サマリー
- `fig2_pr_curve.{png,pdf}` — PR曲線
- `fig3_roc_curve.{png,pdf}` — ROC曲線
- `fig4_cost_threshold.{png,pdf}` — コスト-閾値曲線
- `fig5_distribution_shift.{png,pdf}` — 分布シフト評価

PDF版はLaTeX本文に直接 `\includegraphics` できる。
PNG版はスライド/Notion/Slack 共有用。